# RDKit: substructure queries using SMARTS

This notebook uses **SMARTS** patterns to search molecules for substructures, such as functional groups or ring systems, and uses RDKit's drawing tools to highlight the matches.

By the end you should be able to:

- explain the difference between SMILES and SMARTS
- write simple SMARTS patterns for common functional groups
- test queries against molecules that should and should not match
- check whether a molecule contains a pattern, find where, and count how many times
- search a file of approved drugs for a pattern

Cells marked **Exercise** are for you to try. A hidden solution follows each one - click "Solution" to show it, but have a go first!

## Setup

**Using Google Colab?** Run the cell below first: click it, then press **Shift + Enter** (or click the ▶ button on its left). It installs RDKit and downloads the data this notebook needs, and takes about 30 seconds. On your own computer it does nothing.

In [ ]:
# Setup for Google Colab (does nothing on your own computer)
import os, sys, urllib.request

if 'google.colab' in sys.modules:
    %pip install -q rdkit
    DATA_URL = 'https://raw.githubusercontent.com/dylancjohn/rdkit-tutorials/master/data/'
    os.makedirs('../data', exist_ok=True)
    for name in ['approved_drugs.sdf']:
        if not os.path.exists('../data/' + name):
            urllib.request.urlretrieve(DATA_URL + name, '../data/' + name)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

# uncomment this if the images look low quality
# IPythonConsole.ipython_useSVG=True

# 1. SMILES vs SMARTS

In notebook 1 we used SMILES to describe one specific molecule. SMARTS looks very similar, but describes a **pattern**. One SMARTS pattern can match part of many different molecules.

For example, the SMILES `CCO` is ethanol, while the SMARTS `[OH]` means "an oxygen with one hydrogen", which appears in lots of molecules.

We create a pattern with `Chem.MolFromSmarts()` and check whether a molecule contains it with `HasSubstructMatch()`:

In [ ]:
ethanol = Chem.MolFromSmiles('CCO')
hydroxyl = Chem.MolFromSmarts('[OH]')

ethanol.HasSubstructMatch(hydroxyl)

Some useful SMARTS building blocks:

| SMARTS | Meaning |
|---|---|
| `C` | aliphatic (non-aromatic) carbon |
| `c` | aromatic carbon |
| `[#6]` | any carbon (atomic number 6), aromatic or not |
| `*` | any atom |
| `[N,O]` | nitrogen OR oxygen |
| `[!C]` | anything except an aliphatic carbon |
| `[OH]` | oxygen with exactly one hydrogen |
| `[CX4]` | carbon with 4 connections (hydrogens count too), i.e. an sp3 carbon |
| `[R]` | atom in a ring |
| `[R2]` | atom in exactly 2 rings |
| `[r5]` | atom in a 5-membered ring |
| `-` `=` `#` `:` | single, double, triple, aromatic bond |
| `~` | any bond |

Square brackets hold the properties of one atom, and you can combine them: `[cR2]` means "an aromatic carbon in exactly 2 rings". Atoms written next to each other are bonded, just like in SMILES. If no bond symbol is given, the bond can be single or aromatic.

The full list is in the [Daylight SMARTS documentation](http://www.daylight.com/dayhtml/doc/theory/theory.smarts.html), and RDKit's extensions are described [here](https://rdkit.org/docs/RDKit_Book.html#smarts-support-and-extensions).

## Some example molecules

Our first test molecules are a small set of ring systems:

In [ ]:
names = ['naphthalene', 'benzoxazole', 'indane', 'skatole', 'benzene', 'quinoline']
smiles = ['c12ccccc1cccc2', 'n1c2ccccc2oc1', 'c1ccc2c(c1)CCC2',
          'CC1=CNC2=CC=CC=C12', 'c1ccccc1', 'n1cccc2ccccc12']

# make a molecule from each SMILES string
my_molecules = [Chem.MolFromSmiles(s) for s in smiles]

Draw.MolsToGridImage(my_molecules, legends=names, useSVG=False)

`[R2]` matches an atom that RDKit assigns to two rings. Such atoms occur in fused rings and in spiro systems, so this is a ring-membership query rather than a specific fused-ring query. We check every molecule using a list comprehension, which runs the check on each molecule and collects the answers in a list (in the same order as the grid above):

In [ ]:
in_two_rings = Chem.MolFromSmarts('[R2]')

[mol.HasSubstructMatch(in_two_rings) for mol in my_molecules]

Every molecule has a fused ring except benzene, the fifth one.

A **spiro** system shares one atom between two rings, while fused rings share a bond. The shared spiro atom also matches `[R2]`. This is why that query alone does not specifically identify fused rings:

In [ ]:
spiro = Chem.MolFromSmiles('C1CCC2(CC1)CCCC2')
print(spiro.HasSubstructMatch(in_two_rings))
spiro

### Exercise 1: predict, then check

For each pattern below, write down which of the six molecules you think will match **before** running the cell.

- `[n]` - aromatic nitrogen
- `[r5]` - atom in a 5-membered ring
- `[CR]` - aliphatic (non-aromatic) carbon in a ring

In [ ]:
patterns = ['[n]', '[r5]', '[CR]']

for p in patterns:
    query = Chem.MolFromSmarts(p)
    # zip lets us loop over the names and molecules together
    hits = [name for name, mol in zip(names, my_molecules) if mol.HasSubstructMatch(query)]
    print(p, hits)

<details>
<summary>Solution</summary>

- `[n]`: benzoxazole, skatole and quinoline.
- `[r5]`: benzoxazole, indane and skatole.
- `[CR]`: only indane. Skatole was written with `C` and `=` in its SMILES, but RDKit recognises the ring as aromatic and turns those atoms into `c`. Skatole's methyl carbon is aliphatic but is not in a ring.

</details>

# 2. Where does it match, and how many times?

Let's switch to some familiar drugs:

In [ ]:
drug_names = ['aspirin', 'paracetamol', 'ibuprofen', 'caffeine']
drug_smiles = ['CC(=O)Oc1ccccc1C(=O)O',
               'CC(=O)Nc1ccc(O)cc1',
               'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
               'Cn1cnc2c1c(=O)n(C)c(=O)n2C']

drugs = [Chem.MolFromSmiles(s) for s in drug_smiles]

Draw.MolsToGridImage(drugs, legends=drug_names, useSVG=False)

There are three main functions for substructure searching:

- `HasSubstructMatch(pattern)` - is the pattern there? Returns `True` or `False`.
- `GetSubstructMatch(pattern)` - where is the first match? Returns the atom indices.
- `GetSubstructMatches(pattern)` - where are all the matches? Returns one set of atom indices per match.

In [ ]:
aspirin = drugs[0]
carbonyl = Chem.MolFromSmarts('C=O')

print(aspirin.HasSubstructMatch(carbonyl))
print(aspirin.GetSubstructMatch(carbonyl))
print(aspirin.GetSubstructMatches(carbonyl))

These are the same RDKit atom indices you inspected in notebook 1. Each match is a tuple of atom indices, in the same order as the atoms in the pattern (here: the C, then the O). Aspirin has two C=O groups, so `GetSubstructMatches` finds two matches.

To see which atoms these are, we can turn on atom index labels. After a substructure search, displaying a single molecule highlights the match automatically:

In [ ]:
IPythonConsole.drawOptions.addAtomIndices = True
aspirin.GetSubstructMatches(carbonyl)
aspirin

In [ ]:
# turn the labels off again
IPythonConsole.drawOptions.addAtomIndices = False

To count matches, take the length of the result:

In [ ]:
len(aspirin.GetSubstructMatches(carbonyl))

### Exercise 2: counting

1. How many aromatic carbons does ibuprofen (`drugs[2]`) have?
2. How many oxygen atoms of any kind does aspirin have? (Hint: use the atomic number.)

In [ ]:
# Exercise 2: your code here

<details>
<summary>Solution</summary>

```python
ibuprofen = drugs[2]
print(len(ibuprofen.GetSubstructMatches(Chem.MolFromSmarts('c'))))     # 6

print(len(aspirin.GetSubstructMatches(Chem.MolFromSmarts('[#8]'))))    # 4
```

</details>

# 3. Writing specific patterns

A pattern that is too general will match things you didn't intend. Before searching a large file, decide which chemical group you want and test the query on two kinds of example:

- **Positive examples:** molecules that should match.
- **Negative examples:** similar molecules that should not match.

A query that finds your positive examples but also finds the negative ones needs to be made more specific. A few successful examples do not prove that it covers every possible molecule. Let's test `[OH]` on some small molecules:

In [ ]:
test_names = ['ethanol', 'phenol', 'acetic acid', 'ethylamine', 'acetamide']
test_smiles = ['CCO', 'Oc1ccccc1', 'CC(=O)O', 'CCN', 'CC(N)=O']
test_mols = [Chem.MolFromSmiles(s) for s in test_smiles]

Draw.MolsToGridImage(test_mols, legends=test_names, useSVG=False)

In [ ]:
oh = Chem.MolFromSmarts('[OH]')
[mol.HasSubstructMatch(oh) for mol in test_mols]

If we wanted **alcohols**, `[OH]` is too general: it also matches phenol and acetic acid. We can be more specific by saying what the O is attached to. `[CX4][OX2H]` means "an sp3 carbon bonded to an O which has two connections, one of them a hydrogen":

In [ ]:
alcohol = Chem.MolFromSmarts('[CX4][OX2H]')
[mol.HasSubstructMatch(alcohol) for mol in test_mols]

### Exercise 3

Test our `alcohol` query on propan-2-ol (`CC(O)C`, which should match) and acetic acid (`CC(=O)O`, which should not). Predict the two answers, then create the molecules and run the query.

In [ ]:
# Exercise 3: your code here

<details>
<summary>Solution</summary>

```python
check_mols = [Chem.MolFromSmiles('CC(O)C'), Chem.MolFromSmiles('CC(=O)O')]
[mol.HasSubstructMatch(alcohol) for mol in check_mols]
```

The result should be `[True, False]`. These examples check one intended match and one similar molecule we want to exclude. More examples would be needed for a broader chemical search.
</details>

### Exercise 4: fix the pattern

Someone used `[NH2]` to search for primary amines (R-NH2).

1. Which of the test molecules does it match? Why is that a problem?
2. Write a better pattern that matches ethylamine but not acetamide.

In [ ]:
# Exercise 4: your code here

<details>
<summary>Solution</summary>

`[NH2]` matches ethylamine **and** acetamide. Acetamide is an amide, not an amine, but its N also has two hydrogens.

One fix is to require the N to be attached to an sp3 carbon:

```python
amine = Chem.MolFromSmarts('[NH2][CX4]')
[mol.HasSubstructMatch(amine) for mol in test_mols]
```

A more general version, used in the next section, is `[NX3;H2;!$(NC=O)]`. Here `;` means AND, and `!$(NC=O)` means "not an N attached to a C=O".

</details>

## Watch out for aromaticity

When RDKit reads a SMILES, it works out which rings are aromatic and stores those atoms as aromatic (`c`, `n`, `o`...), even if the SMILES was written with alternating double bonds. This affects matching:

In [ ]:
benzene = Chem.MolFromSmiles('C1=CC=CC=C1')

print(benzene.HasSubstructMatch(Chem.MolFromSmarts('C1=CC=CC=C1')))  # aliphatic C: no match
print(benzene.HasSubstructMatch(Chem.MolFromSmarts('c1ccccc1')))     # aromatic c: match

Caffeine clearly has two C=O groups, but `C=O` doesn't find them. RDKit treats caffeine's rings as aromatic, so the carbonyl carbons are `c`, not `C`. Using `[#6]` (any carbon) fixes this:

In [ ]:
caffeine = drugs[3]

print(len(caffeine.GetSubstructMatches(Chem.MolFromSmarts('C=O'))))
print(len(caffeine.GetSubstructMatches(Chem.MolFromSmarts('[#6]=O'))))

# 4. A functional group finder

Now we can put several patterns together. We store them in a Python **dictionary**, which maps a name (the key) to a value (here, the SMARTS string):

In [ ]:
functional_groups = {
    'alcohol': '[CX4][OX2H]',
    'phenol': 'c[OX2H]',
    'carboxylic acid': '[CX3](=O)[OX2H1]',
    'ester': '[#6][CX3](=O)[OX2H0][#6]',
    'amide': '[NX3][CX3]=[OX1]',
    'primary amine': '[NX3;H2;!$(NC=O)]',
}

Parentheses are branches, just like in SMILES. For example, `[CX3](=O)[OX2H1]` is a carbon with 3 connections that has a double-bonded O and is also bonded to an OH.

This function checks a molecule against every pattern in the dictionary and returns a list of the groups it finds:

In [ ]:
def find_groups(mol):
    found = []
    for group, smarts in functional_groups.items():
        pattern = Chem.MolFromSmarts(smarts)
        if mol.HasSubstructMatch(pattern):
            found.append(group)
    return found


for name, mol in zip(drug_names, drugs):
    print(name, ':', find_groups(mol))

Caffeine finds nothing, because of the aromaticity issue above: its amide-like C=O carbons are aromatic, and our amide pattern asks for `[CX3]` (aliphatic).

We can also highlight one group across all the drugs. `highlightAtomLists` needs one list of atoms per molecule:

In [ ]:
acid = Chem.MolFromSmarts(functional_groups['carboxylic acid'])
highlights = [mol.GetSubstructMatch(acid) for mol in drugs]

Draw.MolsToGridImage(drugs, legends=drug_names, highlightAtomLists=highlights, useSVG=False)

### Exercise 5: add your own groups

Add patterns for a **halogen** (F, Cl, Br or I) and a **nitrile** (C triple-bonded to N) to `functional_groups`, then run `find_groups` on citalopram, which contains both.

In [ ]:
citalopram = Chem.MolFromSmiles('CN(C)CCCC1(OCc2cc(ccc21)C#N)c1ccc(F)cc1')

# Exercise 5: your code here

citalopram

<details>
<summary>Solution</summary>

```python
functional_groups['halogen'] = '[F,Cl,Br,I]'
functional_groups['nitrile'] = 'C#N'

find_groups(citalopram)
```

</details>

# 5. Searching a real dataset

So far we've typed our molecules in by hand. In real work you usually start from a file. The `data` folder contains `approved_drugs.sdf`, which holds 1284 approved drugs from ChEMBL.

An **SD file** (`.sdf`) is a common format for storing many molecules in one file. For each molecule it stores the atoms, bonds and 2D or 3D coordinates, plus any number of named **properties**, such as an ID, a name or a measured value.

RDKit prints a warning whenever it finds a small problem in a file. There are a few harmless ones in this file, so we turn the warnings off to keep the output tidy:

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

To read the file, we use `Chem.SDMolSupplier`. It works just like `SmilesMolSupplier` in notebook 1: it gives us one molecule at a time as we loop over it, and gives `None` for any entry it can't read, so we skip those with `if mol is not None`.

In [ ]:
supplier = Chem.SDMolSupplier('../data/approved_drugs.sdf')
approved = [mol for mol in supplier if mol is not None]

len(approved)

Each item in `approved` is an ordinary RDKit molecule, just like the ones we made from SMILES:

In [ ]:
approved[1]

The properties from the file are stored on each molecule. `GetPropsAsDict()` shows all of them as a dictionary (property name: value):

In [ ]:
approved[1].GetPropsAsDict()

To get just one property, use `GetProp` with the property's name, as we did with `_Name` in notebook 1. We'll use the `name` property to label our drawings:

In [ ]:
approved[1].GetProp('name')

Now we can search the whole list exactly as before. This is the same list comprehension we used on six molecules in section 1; it just runs over 1284 molecules instead:

In [ ]:
acid_hits = [mol for mol in approved if mol.HasSubstructMatch(acid)]
print(len(acid_hits), 'of', len(approved), 'drugs contain a carboxylic acid')

And draw the first 8 hits, labelled with their names:

In [ ]:
first_hits = acid_hits[:8]   # [:8] takes the first 8 items of the list

Draw.MolsToGridImage(first_hits,
                     legends=[mol.GetProp('name') for mol in first_hits],
                     highlightAtomLists=[mol.GetSubstructMatch(acid) for mol in first_hits],
                     molsPerRow=4,
                     subImgSize=(250, 250),
                     useSVG=False)

We can count every group in our dictionary in the same way. `sum()` adds up the True/False results, with True counting as 1:

In [ ]:
for group, smarts in functional_groups.items():
    pattern = Chem.MolFromSmarts(smarts)
    count = sum(mol.HasSubstructMatch(pattern) for mol in approved)
    print(group, count)

### Exercise 6: halogens in drugs

What percentage of the approved drugs contain at least one halogen? Draw six of them with the halogen highlighted.

In [ ]:
# Exercise 6: your code here

<details>
<summary>Solution</summary>

```python
halogen = Chem.MolFromSmarts('[F,Cl,Br,I]')
halogen_hits = [mol for mol in approved if mol.HasSubstructMatch(halogen)]
print(100 * len(halogen_hits) / len(approved), '%')   # about 26 %

six = halogen_hits[:6]
Draw.MolsToGridImage(six,
                     legends=[mol.GetProp('name') for mol in six],
                     highlightAtomLists=[mol.GetSubstructMatch(halogen) for mol in six],
                     subImgSize=(250, 250),
                     useSVG=False)
```

</details>

# 6. Going further (optional)

A few more things you might find useful:

**Stereochemistry.** By default, substructure matching ignores stereochemistry. Pass `useChirality=True` to take it into account:

In [ ]:
l_alanine = Chem.MolFromSmiles('C[C@@H](C(=O)O)N')
d_alanine = Chem.MolFromSmiles('C[C@H](C(=O)O)N')
query = Chem.MolFromSmarts('C[C@@H](C(=O)O)N')

print(d_alanine.HasSubstructMatch(query))                     # True: stereo ignored
print(d_alanine.HasSubstructMatch(query, useChirality=True))  # False
print(l_alanine.HasSubstructMatch(query, useChirality=True))  # True

**Deleting substructures.** `Chem.DeleteSubstructs` removes every match of a pattern from a molecule:

In [ ]:
Chem.DeleteSubstructs(aspirin, acid)

**More examples.** Daylight has a long list of [example SMARTS patterns](https://www.daylight.com/dayhtml_tutorials/languages/smarts/smarts_examples.html) for common functional groups, which is a good place to start when writing your own.

Tutorial originally by Curt Fischer, 2016. Expanded for this workshop.